# Pipeline PROF2 - Small Language Model

Notebook de referência para a entrega final. Ele centraliza os comandos usados no projeto: pré-treino até 2B tokens, mid-training, SFT, avaliação, geração e demo.

Arquivos pesados (`data/*.bin` e `checkpoints/*.pt`) ficam locais e/ou hospedados externamente. O GitHub deve conter código, configs, logs, gráficos, JSONs de avaliação, slides, README e este notebook.

## 1. Ambiente

Use um kernel Python com PyTorch/CUDA. A célula abaixo deve mostrar `cuda disponível: True` para treinar na GPU.

In [ ]:
import sys
import torch

PYTHON = sys.executable

print('python:', sys.executable)
print('torch:', torch.__version__)
print('cuda disponível:', torch.cuda.is_available())
print('gpus:', torch.cuda.device_count())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

## 2. Dados locais

Confere os arquivos binários usados no pré-treino e nas etapas conversacionais.

In [ ]:
from pathlib import Path
import numpy as np

files = [
    ('pretrain train', Path('data/fineweb_edu_train_500m.bin'), np.uint16),
    ('pretrain val', Path('data/fineweb_edu_val_500m.bin'), np.uint16),
    ('pretrain 2b train', Path('data/fineweb_edu_train_2b.bin'), np.uint16),
    ('pretrain 2b val', Path('data/fineweb_edu_val_2b.bin'), np.uint16),
    ('mid train', Path('data/smoltalk_mid_train.bin'), np.uint16),
    ('mid val', Path('data/smoltalk_mid_val.bin'), np.uint16),
    ('sft train tokens', Path('data/smoltalk_sft_train_tokens.bin'), np.uint16),
    ('sft train labels', Path('data/smoltalk_sft_train_labels.bin'), np.int32),
    ('sft val tokens', Path('data/smoltalk_sft_val_tokens.bin'), np.uint16),
    ('sft val labels', Path('data/smoltalk_sft_val_labels.bin'), np.int32),
]

for label, path, dtype in files:
    if not path.exists():
        print(label, 'NÃO ENCONTRADO:', path)
        continue
    arr = np.memmap(path, dtype=dtype, mode='r')
    mb = path.stat().st_size / 1024 / 1024
    print(f'{label}: {len(arr):,} itens | {mb:.2f} MB | {path}')

## 3. Pré-treino original - 60k steps / 491,5M tokens

Esta foi a base da Entrega 1. Rode apenas se precisar refazer ou continuar do checkpoint atual.

In [ ]:
# Recria os dados de pré-treino, se necessário.
# !"{PYTHON}" scripts/prepare_data.py --config configs/pretrain_llama_100m.yaml --max-documents 500000

In [ ]:
# Continua/roda o pré-treino local até 60k steps.
# !"{PYTHON}" src/train.py --config configs/pretrain_llama_100m_rtx3060_60k.yaml --resume checkpoints/ckpt_last.pt

## 4. Pré-treino final para 1B e 2B tokens

Como já temos `fineweb_edu_train_500m.bin`, o correto é preparar somente a fatia adicional de aproximadamente 1,5B tokens, combiná-la com os 500M existentes e gerar `fineweb_edu_train_2b.bin`.

Fluxo usado:

- 1B: continua de `checkpoints/ckpt_last.pt` até 122.100 steps na RTX 3060.
- 2B: continua de `checkpoints/pretrain_1b/ckpt_last.pt` com DDP no servidor PUCRS, usando `configs/pretrain_llama_100m_a5000_2gpu_2b.yaml` até aproximadamente 183.150 steps globais.
- Depois do pré-treino 2B, mid-training e SFT foram refeitos a partir de `checkpoints/pretrain_2b/ckpt_best.pt`.

In [ ]:
# Prepara a fatia nova de ~1,5B tokens e concatena com os 500M existentes.
# Rode apenas se data/fineweb_edu_train_2b.bin e data/fineweb_edu_val_2b.bin ainda não existirem.
# !"{PYTHON}" scripts/prepare_data.py --config configs/pretrain_fineweb_extra_1_5b.yaml --skip-documents 500000 --max-documents 1500000
# !"{PYTHON}" scripts/merge_token_bins.py --output data/fineweb_edu_train_2b.bin --inputs data/fineweb_edu_train_500m.bin data/fineweb_edu_train_extra_1_5b.bin
# !"{PYTHON}" scripts/merge_token_bins.py --output data/fineweb_edu_val_2b.bin --inputs data/fineweb_edu_val_500m.bin data/fineweb_edu_val_extra_1_5b.bin

In [ ]:
# Continua até aproximadamente 1B tokens vistos.
# Esta versão roda dentro do próprio kernel do notebook, sem chamar outro `python`.
from pathlib import Path
import os
import sys
import torch

ROOT = Path(r'E:\trabalhos\Prof2')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))

from train import run_training
from utils import load_config

print('python:', sys.executable)
print('torch:', torch.__version__)
print('cuda disponível:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('CUDA não está disponível neste kernel. Troque o kernel do VS Code antes de treinar.')
print('gpu:', torch.cuda.get_device_name(0))

config = load_config('configs/pretrain_llama_100m_rtx3060_1b.yaml')
run_training(config, resume_path='checkpoints/ckpt_last.pt')

In [ ]:
# Continuação de 1B até aproximadamente 2B tokens no servidor PUCRS.
# Rode em um terminal/tmux no servidor, não no notebook local:
#
# torchrun --standalone --nproc_per_node=2 src/train.py \
#   --config configs/pretrain_llama_100m_a5000_2gpu_2b.yaml \
#   --resume checkpoints/pretrain_1b/ckpt_last.pt
#
# Saídas esperadas:
# - checkpoints/pretrain_2b/ckpt_best.pt
# - checkpoints/pretrain_2b/ckpt_last.pt
# - outputs/train_log_1b_2b.csv

## 5. Mid-training e SFT

Fluxo usado na entrega final com SmolTalk. O mid-training parte do melhor checkpoint de pré-treino 2B, e o SFT parte do melhor checkpoint de mid-training.

In [ ]:
# Preparação dos dados conversacionais.
# !"{PYTHON}" scripts/prepare_chat_data.py --config configs/midtrain_smoltalk.yaml --phase mid --max-examples 50000
# !"{PYTHON}" scripts/prepare_chat_data.py --config configs/sft_smoltalk.yaml --phase sft --max-examples 50000

In [ ]:
# Mid-training a partir do melhor pré-treino 2B.
# !"{PYTHON}" src/finetune.py --config configs/midtrain_smoltalk.yaml --init-checkpoint checkpoints/pretrain_2b/ckpt_best.pt

# SFT a partir do melhor checkpoint de mid-training.
# !"{PYTHON}" src/finetune.py --config configs/sft_smoltalk.yaml --init-checkpoint checkpoints/midtrain_best.pt

## 6. Avaliação e curvas

Gera curvas, perplexidade e benchmark de múltipla escolha para os artefatos da entrega.

In [ ]:
!"{PYTHON}" src/plot_loss.py --log outputs/train_log_1b_2b.csv --output outputs/loss_curve_1b_2b.png
!"{PYTHON}" scripts/plot_pretrain_combined.py --logs outputs/train_log_0_60k.csv outputs/train_log_60k_1b.csv outputs/train_log_1b_2b.csv --plot-output outputs/pretrain_loss_curve_0_2b.png --json-output outputs/pretrain_val_curve_0_2b.json
!"{PYTHON}" src/plot_loss.py --log outputs/midtrain_log.csv --output outputs/midtrain_loss_curve.png
!"{PYTHON}" src/plot_loss.py --log outputs/sft_log.csv --output outputs/sft_loss_curve.png

In [ ]:
!"{PYTHON}" src/evaluate.py val_curve --log outputs/train_log_1b_2b.csv --output outputs/pretrain_val_curve.json
!"{PYTHON}" src/evaluate.py perplexity --checkpoint checkpoints/sft_best.pt --data data/fineweb_edu_val_2b.bin --eval-iters 100 --batch-size 1 --output outputs/sft_perplexity.json
!"{PYTHON}" src/evaluate.py multiple_choice --checkpoint checkpoints/sft_best.pt --benchmark hellaswag --split validation --max-examples 200 --output outputs/sft_hellaswag_200.json

## 7. Geração e demo

Use geração simples para sanity check e Streamlit para a demo final.

In [ ]:
!"{PYTHON}" src/generate.py --checkpoint checkpoints/sft_best.pt --prompt 'User:\nWhat is the importance of study?\nAssistant:\n' --max_new_tokens 120 --temperature 0.7 --top_k 50 --output outputs/samples_sft_final.txt

In [ ]:
# Rode no terminal do VS Code, não dentro do notebook:
# python -m streamlit run app.py

## 8. O que salvar no GitHub e o que não salvar

Salvar no GitHub:

- código em `src/`, `scripts/` e `app.py`;
- configs em `configs/`;
- `README.md`, `Treino.ipynb`, `requirements.txt`;
- logs, curvas e JSONs de avaliação em `outputs/`;
- slides em `slides/`.

Não salvar no GitHub:

- `data/*.bin`;
- `checkpoints/*.pt`;
- `server_backup/`;
- caches `__pycache__/` e `.vscode/`.

Hospedar externamente para a entrega:

- `checkpoints/sft_best.pt` é obrigatório;
- `checkpoints/pretrain_2b/ckpt_best.pt` e `checkpoints/midtrain_best.pt` são recomendados.